# Lab Test Booking Assistant - Agent Project

A chat assistant for a fictional lab, **CityLab Diagnostics**. It lists tests, checks slots, books and cancels
(only after you approve), answers general health questions with web search, and never diagnoses.

**How to run (VS Code on Windows)**
1. Open this notebook in VS Code and pick a Python 3.10+ kernel (top right). If VS Code asks to install `ipykernel`, click Install.
2. Get a free Groq API key at https://console.groq.com/keys. Set it as the environment variable `GROQ_API_KEY`,
   or paste it when the setup cell asks (a box opens at the top of VS Code).
3. Run the cells from top to bottom (Run All works too). The database `lab.db` is created next to the notebook.

Groq's free tier has a tokens-per-minute limit, so some cells may pause for a few seconds and then continue. That's normal.

In [1]:
# Run once. If imports fail right after installing, restart the kernel and continue from the next cell.
%pip install -q "langchain>=1.5,<2" "langgraph>=1.2,<2" "langchain-groq>=1.2,<2" ddgs pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Setup

In [2]:
import os, json, sqlite3, time
from contextlib import closing
from dataclasses import dataclass
from datetime import date, timedelta
from getpass import getpass

import pandas as pd
from ddgs import DDGS
from langchain.agents import create_agent
from langchain.agents.middleware import (HumanInTheLoopMiddleware, ModelRetryMiddleware,
                                         PIIMiddleware, ToolRetryMiddleware, wrap_model_call)
from langchain.tools import tool, ToolRuntime
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Paste your Groq API key: ")

MODEL = "openai/gpt-oss-120b"   # Groq model (llama-3.3-70b was retired in Aug 2026). If you hit the daily
                                # limit, try "openai/gpt-oss-20b" (it has its own quota).
DB = "lab.db"                   # SQLite file, created next to this notebook
print("Setup done")

Setup done


## 1. Database
Three tables: `tests`, `slots`, `bookings`. 10 tests in 3 categories, 3 slots each.
**MRI Brain** has no free slots. `CREATE TABLE IF NOT EXISTS` + `INSERT OR IGNORE` make the setup safe to run twice.

In [3]:
TESTS = [  # (name, category, price in Rs)
    ("Complete Blood Count", "Blood", 300),
    ("Lipid Profile", "Blood", 600),
    ("Blood Sugar Fasting", "Blood", 150),
    ("HbA1c", "Blood", 450),
    ("Thyroid Profile", "Hormone", 500),
    ("Vitamin D", "Hormone", 1200),
    ("Vitamin B12", "Hormone", 900),
    ("Chest X-Ray", "Imaging", 400),
    ("Ultrasound Abdomen", "Imaging", 1500),
    ("MRI Brain", "Imaging", 6500),
]
FULL_TEST = "MRI Brain"   # all its slots are already taken


def db():
    con = sqlite3.connect(DB)
    con.row_factory = sqlite3.Row
    return con


def q(sql, params=()):
    """Run a SELECT and show it as a table. Only for proofs in this notebook, the model never uses it."""
    with closing(sqlite3.connect(DB)) as con:
        return pd.read_sql_query(sql, con, params=params)


def setup_db(fresh=False):
    tomorrow = date.today() + timedelta(days=1)
    times = [f"{tomorrow} 09:00", f"{tomorrow} 12:00", f"{tomorrow + timedelta(days=1)} 10:00"]
    with closing(db()) as con:
        if fresh:
            con.executescript("DROP TABLE IF EXISTS bookings; DROP TABLE IF EXISTS slots; DROP TABLE IF EXISTS tests;")
        con.executescript("""
            CREATE TABLE IF NOT EXISTS tests (
                id INTEGER PRIMARY KEY,
                name TEXT UNIQUE NOT NULL,
                category TEXT NOT NULL,
                price INTEGER NOT NULL);
            CREATE TABLE IF NOT EXISTS slots (
                id INTEGER PRIMARY KEY,
                test_id INTEGER NOT NULL REFERENCES tests(id),
                slot_time TEXT NOT NULL,
                is_booked INTEGER NOT NULL DEFAULT 0,
                UNIQUE (test_id, slot_time));
            CREATE TABLE IF NOT EXISTS bookings (
                id INTEGER PRIMARY KEY,
                user_id TEXT NOT NULL,
                test_id INTEGER NOT NULL REFERENCES tests(id),
                slot_id INTEGER NOT NULL REFERENCES slots(id),
                status TEXT NOT NULL DEFAULT 'booked',
                created_at TEXT DEFAULT CURRENT_TIMESTAMP);
        """)
        con.executemany("INSERT OR IGNORE INTO tests (name, category, price) VALUES (?, ?, ?)", TESTS)
        for name, _, _ in TESTS:
            test_id = con.execute("SELECT id FROM tests WHERE name = ?", (name,)).fetchone()["id"]
            for t in times:
                con.execute("INSERT OR IGNORE INTO slots (test_id, slot_time, is_booked) VALUES (?, ?, ?)",
                            (test_id, t, int(name == FULL_TEST)))
        con.commit()

In [4]:
setup_db(fresh=True)   # start clean
setup_db()             # run it again on purpose...
setup_db()             # ...and again: still no duplicates
print(q("SELECT (SELECT COUNT(*) FROM tests) AS tests, (SELECT COUNT(*) FROM slots) AS slots"))

q("""SELECT t.id, t.category, t.name, t.price, COUNT(s.id) AS slots, SUM(s.is_booked = 0) AS free
     FROM tests t JOIN slots s ON s.test_id = t.id
     GROUP BY t.id ORDER BY t.id""")

   tests  slots
0     10     30


,id,category,name,price,slots,free
0,1,Blood,Complete Blood Count,300,3,3
1,2,Blood,Lipid Profile,600,3,3
2,3,Blood,Blood Sugar Fasting,150,3,3
3,4,Blood,HbA1c,450,3,3
4,5,Hormone,Thyroid Profile,500,3,3
5,6,Hormone,Vitamin D,1200,3,3
6,7,Hormone,Vitamin B12,900,3,3
7,8,Imaging,Chest X-Ray,400,3,3
8,9,Imaging,Ultrasound Abdomen,1500,3,3
9,10,Imaging,MRI Brain,6500,3,0


## 2. Read tools
`list_tests`, `check_slots`, `my_bookings`. The model only passes simple values (a test name);
all SQL is written here with `?` placeholders.

In [5]:
@dataclass
class Ctx:
    user_id: str   # set by the app for every run, never by the model


def find_test(con, name):
    """Find one test by (part of) its name. Returns the row, or an error message."""
    name = name.strip()
    rows = con.execute("SELECT * FROM tests WHERE name LIKE ?", (f"%{name}%",)).fetchall()
    exact = [r for r in rows if r["name"].lower() == name.lower()]
    if exact or len(rows) == 1:
        return (exact or rows)[0]
    if not rows:
        return f"No test called '{name}'. Use list_tests to see all tests."
    return "Several tests match: " + ", ".join(r["name"] for r in rows) + ". Which one?"


@tool
def list_tests(category: str = "") -> str:
    """List the lab's tests with price and number of free slots. Optional: filter by category."""
    with closing(db()) as con:
        rows = con.execute("""
            SELECT t.name, t.category, t.price,
                   (SELECT COUNT(*) FROM slots s WHERE s.test_id = t.id AND s.is_booked = 0) AS free
            FROM tests t WHERE t.category LIKE ? ORDER BY t.category, t.name""", (f"%{category}%",)).fetchall()
    if not rows:
        return "No tests found."
    return "\n".join(f"{r['name']} ({r['category']}) - Rs.{r['price']} - {r['free']} free slots" for r in rows)


FAIL_NEXT_CHECK = False   # section 8 sets this to True to simulate one failure
check_slots_calls = 0


@tool
def check_slots(test_name: str) -> str:
    """Show the free time slots (with slot_id) for one test."""
    global FAIL_NEXT_CHECK, check_slots_calls
    check_slots_calls += 1
    if FAIL_NEXT_CHECK:
        FAIL_NEXT_CHECK = False
        print("   check_slots -> FAILED (simulated database timeout)")
        raise TimeoutError("database timeout")
    with closing(db()) as con:
        test = find_test(con, test_name)
        if isinstance(test, str):
            return test
        rows = con.execute("SELECT id, slot_time FROM slots WHERE test_id = ? AND is_booked = 0 ORDER BY slot_time",
                           (test["id"],)).fetchall()
    if not rows:
        return f"No free slots for {test['name']}."
    lines = [f"slot_id {r['id']}: {r['slot_time']}" for r in rows]
    return f"Free slots for {test['name']} (Rs.{test['price']}):\n" + "\n".join(lines)


def get_bookings(user_id):
    with closing(db()) as con:
        rows = con.execute("""
            SELECT b.id, t.name, s.slot_time, b.status FROM bookings b
            JOIN tests t ON t.id = b.test_id JOIN slots s ON s.id = b.slot_id
            WHERE b.user_id = ? ORDER BY b.id""", (user_id,)).fetchall()
    if not rows:
        return "You have no bookings."
    return "\n".join(f"booking #{r['id']}: {r['name']} at {r['slot_time']} ({r['status']})" for r in rows)


@tool
def my_bookings(runtime: ToolRuntime[Ctx]) -> str:
    """Show the current user's bookings."""
    return get_bookings(runtime.context.user_id)

In [6]:
print(list_tests.invoke({}))
print()
print(check_slots.invoke({"test_name": "thyroid"}))
print()
print(check_slots.invoke({"test_name": "MRI Brain"}))   # the fully booked test
print()
print(get_bookings("u1"))

Blood Sugar Fasting (Blood) - Rs.150 - 3 free slots
Complete Blood Count (Blood) - Rs.300 - 3 free slots
HbA1c (Blood) - Rs.450 - 3 free slots
Lipid Profile (Blood) - Rs.600 - 3 free slots
Thyroid Profile (Hormone) - Rs.500 - 3 free slots
Vitamin B12 (Hormone) - Rs.900 - 3 free slots
Vitamin D (Hormone) - Rs.1200 - 3 free slots
Chest X-Ray (Imaging) - Rs.400 - 3 free slots
MRI Brain (Imaging) - Rs.6500 - 0 free slots
Ultrasound Abdomen (Imaging) - Rs.1500 - 3 free slots

Free slots for Thyroid Profile (Rs.500):
slot_id 13: 2026-11-19 09:00
slot_id 14: 2026-11-19 12:00
slot_id 15: 2026-11-20 10:00

No free slots for MRI Brain.

You have no bookings.


In [7]:
# Adding a test or slot = new rows only, no code change. The tools pick it up straight away.
with closing(db()) as con:
    con.execute("INSERT OR IGNORE INTO tests (name, category, price) VALUES ('Kidney Function Test', 'Blood', 700)")
    con.execute("INSERT OR IGNORE INTO slots (test_id, slot_time) SELECT id, ? FROM tests WHERE name = 'Kidney Function Test'",
                (f"{date.today() + timedelta(days=2)} 08:30",))
    con.commit()
print(check_slots.invoke({"test_name": "kidney"}))

Free slots for Kidney Function Test (Rs.700):
slot_id 31: 2026-11-20 08:30


## 3. Write tools
`book_test` and `cancel_booking` check: test exists, slot belongs to the test and is free, no duplicate booking,
and the booking belongs to the user. After booking, a receipt is printed from the saved database row.

In [8]:
def receipt(con, booking_id):
    r = con.execute("""
        SELECT b.id, b.user_id, b.status, b.created_at, t.name, t.price, s.slot_time
        FROM bookings b JOIN tests t ON t.id = b.test_id JOIN slots s ON s.id = b.slot_id
        WHERE b.id = ?""", (booking_id,)).fetchone()
    return (f"========== RECEIPT ==========\n"
            f"Booking : #{r['id']} ({r['status']})\n"
            f"User    : {r['user_id']}\n"
            f"Test    : {r['name']}\n"
            f"Slot    : {r['slot_time']}\n"
            f"Price   : Rs.{r['price']}\n"
            f"Created : {r['created_at']}\n"
            f"=============================")


def book(user_id, test_name, slot_id):
    with closing(db()) as con:
        test = find_test(con, test_name)
        if isinstance(test, str):
            return test
        slot = con.execute("SELECT * FROM slots WHERE id = ?", (slot_id,)).fetchone()
        if slot is None or slot["test_id"] != test["id"]:
            return f"slot_id {slot_id} is not a slot of {test['name']}. Use check_slots."
        dup = con.execute("SELECT id FROM bookings WHERE user_id = ? AND test_id = ? AND status = 'booked'",
                          (user_id, test["id"])).fetchone()
        if dup:
            return f"You already have booking #{dup['id']} for {test['name']}."
        # only take the slot if it is still free (no double booking)
        if con.execute("UPDATE slots SET is_booked = 1 WHERE id = ? AND is_booked = 0", (slot_id,)).rowcount == 0:
            return f"Sorry, slot_id {slot_id} is already taken. Use check_slots for free slots."
        booking_id = con.execute("INSERT INTO bookings (user_id, test_id, slot_id) VALUES (?, ?, ?)",
                                 (user_id, test["id"], slot_id)).lastrowid
        con.commit()
        text = receipt(con, booking_id)   # read back from the database row
    print(text)
    return "Booked.\n" + text


def cancel(user_id, booking_id):
    with closing(db()) as con:
        b = con.execute("SELECT * FROM bookings WHERE id = ? AND user_id = ?", (booking_id, user_id)).fetchone()
        if b is None:
            return f"No booking #{booking_id} found for you."
        if b["status"] == "cancelled":
            return f"Booking #{booking_id} is already cancelled."
        con.execute("UPDATE bookings SET status = 'cancelled' WHERE id = ?", (booking_id,))
        con.execute("UPDATE slots SET is_booked = 0 WHERE id = ?", (b["slot_id"],))
        con.commit()
    return f"Booking #{booking_id} cancelled. The slot is free again."


@tool
def book_test(test_name: str, slot_id: int, runtime: ToolRuntime[Ctx]) -> str:
    """Book a test for the current user. Needs the test name and a free slot_id from check_slots."""
    return book(runtime.context.user_id, test_name, slot_id)


@tool
def cancel_booking(booking_id: int, runtime: ToolRuntime[Ctx]) -> str:
    """Cancel one of the current user's bookings by its booking id (get it from my_bookings)."""
    return cancel(runtime.context.user_id, booking_id)

In [9]:
free = q("""SELECT s.id FROM slots s JOIN tests t ON t.id = s.test_id
            WHERE t.name = 'Thyroid Profile' AND s.is_booked = 0 ORDER BY s.id""")["id"].tolist()

print("1) Valid booking:")
book("u1", "Thyroid Profile", free[0])
bid = int(q("SELECT MAX(id) AS id FROM bookings")["id"][0])

print("\n2) Duplicate      ->", book("u1", "Thyroid Profile", free[1]))
print("3) Slot taken     ->", book("u2", "Thyroid Profile", free[0]))
print("4) No such test   ->", book("u2", "Pizza Test", free[1]))
print("5) Wrong test     ->", book("u2", "Lipid Profile", free[1]))
print("6) Not your booking ->", cancel("u2", bid))
print("7) Own booking    ->", cancel("u1", bid))

q("SELECT * FROM bookings")

1) Valid booking:
========== RECEIPT ==========
Booking : #1 (booked)
User    : u1
Test    : Thyroid Profile
Slot    : 2026-11-19 09:00
Price   : Rs.500
Created : 2026-11-18 04:32:13

2) Duplicate      -> You already have booking #1 for Thyroid Profile.
3) Slot taken     -> Sorry, slot_id 13 is already taken. Use check_slots for free slots.
4) No such test   -> No test called 'Pizza Test'. Use list_tests to see all tests.
5) Wrong test     -> slot_id 14 is not a slot of Lipid Profile. Use check_slots.
6) Not your booking -> No booking #1 found for you.
7) Own booking    -> Booking #1 cancelled. The slot is free again.


,id,user_id,test_id,slot_id,status,created_at
0,1,u1,5,13,cancelled,2026-11-18 04:32:13


## 4. User identity
`user_id` comes from the run context (`Ctx`), passed by the app with `agent.invoke(..., context=Ctx(user_id=...))`.
The tools read it from `runtime.context`. Below is the exact schema sent to the model: no `user_id` in it.

In [10]:
DB_TOOLS = [list_tests, check_slots, my_bookings, book_test, cancel_booking]

for t in DB_TOOLS:
    props = convert_to_openai_tool(t)["function"]["parameters"].get("properties", {})
    print(f"{t.name:15} model can set: {list(props)}")

print("\nFull schema the model sees for book_test:")
print(json.dumps(convert_to_openai_tool(book_test), indent=2))

all_schemas = json.dumps([convert_to_openai_tool(t) for t in DB_TOOLS])
print("\n'user_id' visible to the model?", "user_id" in all_schemas)

list_tests      model can set: ['category']
check_slots     model can set: ['test_name']
my_bookings     model can set: []
book_test       model can set: ['test_name', 'slot_id']
cancel_booking  model can set: ['booking_id']

Full schema the model sees for book_test:
{
  "type": "function",
  "function": {
    "name": "book_test",
    "description": "Book a test for the current user. Needs the test name and a free slot_id from check_slots.",
    "parameters": {
      "properties": {
        "test_name": {
          "type": "string"
        },
        "slot_id": {
          "type": "integer"
        }
      },
      "required": [
        "test_name",
        "slot_id"
      ],
      "type": "object"
    }
  }
}

'user_id' visible to the model? False


## 5. Web search
DuckDuckGo search (no extra key). Only for general info. Questions about prices or slots are refused,
because those come from the database only.

In [11]:
@tool
def web_search(query: str) -> str:
    """Search the web for GENERAL health info only, e.g. fasting rules or what a test measures.
    Never use it for prices, slots or a diagnosis."""
    if any(w in query.lower() for w in ["price", "cost", "slot", "availab"]):
        return "Not allowed: prices and slots must come from list_tests and check_slots."
    try:
        hits = DDGS().text(query, max_results=3)
    except Exception as e:
        return f"Web search failed ({e}). Tell the user you could not look it up."
    return "\n".join(f"- {h['title']}: {h['body']}" for h in hits) or "No results."


print(web_search.invoke({"query": "do I need to fast before a lipid profile test"})[:800])
print()
print(web_search.invoke({"query": "lipid profile test price"}))

- Fasting before a cholesterol test: What to do and what to expect: Oct 25, 2018 · Here, learn more about when people should fast before a cholesterol test, how long to fast for, and what the test results mean.
- Fasting Before a Cholesterol Test: Should You Do It? - Healthline: Jun 4, 2024 · Your blood will likely be checked using a total lipid profile test. To understand your results, you’ll need to know the different types of cholesterol that the test measures and what’s ...
- Fasting before lipid test usually unnecessary - Harvard Health: Apr 1, 2025 · Most people don't need to fast before a lipid test because the difference between LDL cholesterol values when fasting or not fasting is very small. But people who have had high triglyceride levels should fast prior to the test.

Not allowed: prices and slots must come from list_tests and check_slots.


## 6. Agent
`create_agent` with a Groq model, a system prompt, and memory (`InMemorySaver`, one thread per conversation).
The middleware list also adds approval (section 7), retry and email redaction (section 8).

In [12]:
SYSTEM_PROMPT = """You are the booking assistant of CityLab Diagnostics, a diagnostic lab.

Rules:
- Tests, prices and free slots come ONLY from list_tests and check_slots. Never guess them and never use web_search for them.
- Confirm before booking: call book_test or cancel_booking only when the user clearly asked for it
  (which test, and a slot or "earliest"). Always call check_slots first to get a real free slot_id.
  Do not ask again in chat: the app shows the user an approve/reject prompt before anything is saved.
  If the user rejects, do not try again.
- To cancel, call my_bookings first to get the booking id.
- For general health questions (fasting, what a test measures) always use web_search,
  and say it is general information, not medical advice.
- Never diagnose and never choose a test from symptoms. If the user describes symptoms,
  tell them to see a doctor and do not book anything.
- Keep answers short."""

TOOLS = [list_tests, check_slots, my_bookings, book_test, cancel_booking, web_search]

model_inputs = []   # the last messages sent to the model (proof for section 8)


@wrap_model_call
def remember_model_input(request, handler):
    model_inputs[:] = request.messages
    return handler(request)


# reasoning_effort is for gpt-oss models; remove it if you switch to a non-reasoning model
llm = ChatGroq(model=MODEL, temperature=0, max_retries=5, reasoning_effort="low")

agent = create_agent(
    model=llm,
    tools=TOOLS,
    system_prompt=SYSTEM_PROMPT,
    context_schema=Ctx,                # user_id comes from here (section 4)
    checkpointer=InMemorySaver(),      # memory across turns
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),                       # section 8
        ToolRetryMiddleware(max_retries=2, tools=["check_slots"], initial_delay=0.5, jitter=False),  # section 8
        ModelRetryMiddleware(max_retries=2, initial_delay=5),                                   # Groq hiccups
        HumanInTheLoopMiddleware(interrupt_on={                                                 # section 7
            "book_test": {"allowed_decisions": ["approve", "reject"]},
            "cancel_booking": {"allowed_decisions": ["approve", "reject"]},
        }),
        remember_model_input,
    ],
)
print("Agent ready. Tools:", [t.name for t in TOOLS])

Agent ready. Tools: ['list_tests', 'check_slots', 'my_bookings', 'book_test', 'cancel_booking', 'web_search']


In [13]:
def chat(user_id, text, approve=None, thread=None, ask=input, echo=True):
    """One turn as user_id. approve=True/False answers approval prompts automatically, None asks you."""
    config = {"configurable": {"thread_id": thread or user_id}}
    ctx = Ctx(user_id=user_id)
    start = len(agent.get_state(config).values.get("messages", []))
    if echo:
        print(f"\n[{user_id}] You: {text}")

    result = agent.invoke({"messages": [{"role": "user", "content": text}]}, config, context=ctx)
    tools_used, shown = [], start
    while True:
        # print the new tool calls / results
        for m in result["messages"][shown:]:
            if m.type == "ai":
                for tc in m.tool_calls:
                    tools_used.append(tc["name"])
                    print(f"  [tool call] {tc['name']} {tc['args']}")
            elif m.type == "tool":
                short = str(m.content).replace("\n", " | ")[:160]
                print(f"  [tool result] {short}")
        shown = len(result["messages"])

        if "__interrupt__" not in result:
            break
        # the agent paused before book_test / cancel_booking and needs approval
        decisions = []
        for req in result["__interrupt__"][0].value["action_requests"]:
            if approve is None:
                ok = ask(f"  Approve {req['name']} {req['args']}? (y/n): ").strip().lower().startswith("y")
            else:
                ok = approve
            print(f"  [approval] {req['name']} -> {'APPROVED' if ok else 'REJECTED'}")
            decisions.append({"type": "approve"} if ok else
                             {"type": "reject", "message": "The user rejected this action. Do not retry it."})
        result = agent.invoke(Command(resume={"decisions": decisions}), config, context=ctx)

    answer = result["messages"][-1].content
    print(f"Assistant: {answer}")
    return tools_used, answer

In [14]:
# Memory across turns: the 2nd question only makes sense if the agent remembers the 1st.
chat("u1", "Hi! What Hormone tests do you have?", thread="memory-demo")
chat("u1", "Which one of those is the cheapest?", thread="memory-demo");


[u1] You: Hi! What Hormone tests do you have?
  [tool call] list_tests {'category': 'Hormone'}
  [tool result] Thyroid Profile (Hormone) - Rs.500 - 3 free slots | Vitamin B12 (Hormone) - Rs.900 - 3 free slots | Vitamin D (Hormone) - Rs.1200 - 3 free slots
Assistant: Here are the hormone‑related tests we offer:

- **Thyroid Profile (Hormone)** – Rs. 500 (3 free slots)  
- **Vitamin B12 (Hormone)** – Rs. 900 (3 free slots)  
- **Vitamin D (Hormone)** – Rs. 1200 (3 free slots)  

Let me know if you’d like more details about any test or want to book a slot.

[u1] You: Which one of those is the cheapest?
Assistant: The **Thyroid Profile (Hormone)** test is the cheapest at **Rs 500**. Let me know if you’d like to see its available slots or book it.


## 7. Approval
Only `book_test` and `cancel_booking` pause for approval. One approve, one reject, then a query proves
the reject changed nothing.

In [15]:
# APPROVE
chat("u1", "Book the earliest free slot for Vitamin B12. Yes, I confirm.", approve=True, thread="approve-demo")
q("SELECT * FROM bookings WHERE status = 'booked'")


[u1] You: Book the earliest free slot for Vitamin B12. Yes, I confirm.
  [tool call] check_slots {'test_name': 'Vitamin B12'}
  [tool result] Free slots for Vitamin B12 (Rs.900): | slot_id 19: 2026-11-19 09:00 | slot_id 20: 2026-11-19 12:00 | slot_id 21: 2026-11-20 10:00
  [tool call] book_test {'slot_id': 19, 'test_name': 'Vitamin B12'}
  [approval] book_test -> APPROVED
========== RECEIPT ==========
Booking : #2 (booked)
User    : u1
Test    : Vitamin B12
Slot    : 2026-11-19 09:00
Price   : Rs.900
Created : 2026-11-18 04:32:23
  [tool result] Booked. | ========== RECEIPT ========== | Booking : #2 (booked) | User    : u1 | Test    : Vitamin B12 | Slot    : 2026-11-19 09:00 | Price   : Rs.900 | Created
Assistant: Your Vitamin B12 test has been booked for **2026-11-19 at 09:00**. Let me know if you need anything else!


,id,user_id,test_id,slot_id,status,created_at
0,2,u1,7,19,booked,2026-11-18 04:32:23


In [16]:
# REJECT, then prove the database did not change
def snapshot():
    with closing(sqlite3.connect(DB)) as con:
        return con.execute("SELECT * FROM bookings").fetchall(), con.execute("SELECT * FROM slots").fetchall()


before = snapshot()
chat("u1", "Book the earliest free slot for Lipid Profile. Yes, I confirm.", approve=False, thread="reject-demo")
after = snapshot()

print("\nBookings rows before:", len(before[0]), "| after:", len(after[0]))
print("Database unchanged after reject:", before == after)
q("""SELECT s.id AS slot_id, s.slot_time, s.is_booked FROM slots s
     JOIN tests t ON t.id = s.test_id WHERE t.name = 'Lipid Profile'""")


[u1] You: Book the earliest free slot for Lipid Profile. Yes, I confirm.
  [tool call] check_slots {'test_name': 'Lipid Profile'}
  [tool result] Free slots for Lipid Profile (Rs.600): | slot_id 4: 2026-11-19 09:00 | slot_id 5: 2026-11-19 12:00 | slot_id 6: 2026-11-20 10:00
  [tool call] book_test {'slot_id': 4, 'test_name': 'Lipid Profile'}
  [approval] book_test -> REJECTED
  [tool result] User rejected the tool call for `book_test` with reason: The user rejected this action. Do not retry it.
Assistant: Your booking was not completed. If you’d like to try again or choose a different slot, just let me know!

Bookings rows before: 2 | after: 2
Database unchanged after reject: True


,slot_id,slot_time,is_booked
0,4,2026-11-19 09:00,0
1,5,2026-11-19 12:00,0
2,6,2026-11-20 10:00,0


## 8. Reliability and privacy
**Retry:** `check_slots` is forced to fail once; `ToolRetryMiddleware` retries it and the agent still answers.
**Privacy:** `PIIMiddleware` replaces emails with `[REDACTED_EMAIL]` before the model sees the message.

In [17]:
FAIL_NEXT_CHECK = True
calls_before = check_slots_calls

chat("u2", "What free slots do you have for Vitamin D?", thread="retry-demo")
print(f"\ncheck_slots ran {check_slots_calls - calls_before} times: 1 simulated failure + 1 automatic retry that worked.")


[u2] You: What free slots do you have for Vitamin D?
   check_slots -> FAILED (simulated database timeout)
  [tool call] check_slots {'test_name': 'Vitamin D'}
  [tool result] Free slots for Vitamin D (Rs.1200): | slot_id 16: 2026-11-19 09:00 | slot_id 17: 2026-11-19 12:00 | slot_id 18: 2026-11-20 10:00
Assistant: Here are the available slots for the Vitamin D test (₹1,200 each):

- **Slot 16:** 19 Nov 2026, 09:00  
- **Slot 17:** 19 Nov 2026, 12:00  
- **Slot 18:** 20 Nov 2026, 10:00  

Let me know which slot you’d like to book.

check_slots ran 2 times: 1 simulated failure + 1 automatic retry that worked.


In [18]:
email = "priya.sharma@example.com"
chat("u2", f"Hi, my email is {email}. Which Imaging tests do you have?", thread="pii-demo")

seen = [m.content for m in model_inputs if m.type == "human"]
print("\nWhat the model actually received:", seen)
print("Email hidden from the model:", all(email not in s for s in seen))


[u2] You: Hi, my email is priya.sharma@example.com. Which Imaging tests do you have?
  [tool call] list_tests {'category': 'Imaging'}
  [tool result] Chest X-Ray (Imaging) - Rs.400 - 3 free slots | MRI Brain (Imaging) - Rs.6500 - 0 free slots | Ultrasound Abdomen (Imaging) - Rs.1500 - 3 free slots
Assistant: Here are the imaging tests we currently offer:

- **Chest X‑Ray** – Rs. 400 (3 slots available)  
- **MRI Brain** – Rs. 6500 (currently fully booked)  
- **Ultrasound Abdomen** – Rs. 1500 (3 slots available)  

Let me know if you’d like to book any of these, or if you have any other questions!

What the model actually received: ['Hi, my email is [REDACTED_EMAIL]. Which Imaging tests do you have?']
Email hidden from the model: True


## 9. Interactive loop
Type messages until `exit`. Type `user u2` to switch user. Each turn prints tool calls, approvals and the answer.
First a scripted run (so Run All works without typing), then proof that one user cannot see or cancel
another user's booking. Set `INTERACTIVE = True` in the last cell to chat yourself.

In [19]:
def chat_loop(script=None, auto_approve=None):
    """Chat until 'exit'. 'user <id>' switches user. script = list of lines to type automatically."""
    lines = list(script) if script else None

    def ask(prompt):
        if lines is None:
            return input(prompt)
        line = lines.pop(0)
        print(prompt + line)
        return line

    user = "u1"
    print("Commands: 'user <id>' switches user, 'exit' quits.")
    while True:
        text = ask(f"\n[{user}] You: ").strip()
        if text.lower() == "exit":
            print("Bye!")
            break
        if text.lower().startswith("user "):
            user = text.split()[1]
            print(f"-- now chatting as {user}")
        elif text:
            chat(user, text, approve=auto_approve, ask=ask, echo=False)

In [20]:
chat_loop([
    "Book the earliest free slot for Chest X-Ray. I confirm.",
    "Show my bookings",
    "user u2",
    "Show my bookings",    # u2 must NOT see u1's bookings
    "exit",
], auto_approve=True)

Commands: 'user <id>' switches user, 'exit' quits.

[u1] You: Book the earliest free slot for Chest X-Ray. I confirm.
  [tool call] check_slots {'test_name': 'Chest X-Ray'}
  [tool result] Free slots for Chest X-Ray (Rs.400): | slot_id 22: 2026-11-19 09:00 | slot_id 23: 2026-11-19 12:00 | slot_id 24: 2026-11-20 10:00
  [tool call] book_test {'slot_id': 22, 'test_name': 'Chest X-Ray'}
  [approval] book_test -> APPROVED
========== RECEIPT ==========
Booking : #3 (booked)
User    : u1
Test    : Chest X-Ray
Slot    : 2026-11-19 09:00
Price   : Rs.400
Created : 2026-11-18 04:32:28
  [tool result] Booked. | ========== RECEIPT ========== | Booking : #3 (booked) | User    : u1 | Test    : Chest X-Ray | Slot    : 2026-11-19 09:00 | Price   : Rs.400 | Created
Assistant: Your Chest X‑Ray has been booked for **2026‑11‑19 at 09:00**. Let me know if you need anything else!

[u1] You: Show my bookings
  [tool call] my_bookings {}
  [tool result] booking #1: Thyroid Profile at 2026-11-19 09:00 (cancel

In [21]:
# u2 tries to cancel u1's booking (even with approval it must fail)
u1_booking = int(q("SELECT id FROM bookings WHERE user_id = 'u1' AND status = 'booked' ORDER BY id DESC LIMIT 1")["id"][0])
print("u1's booking id:", u1_booking)

chat("u2", f"Please cancel booking #{u1_booking}. Yes, I confirm.", approve=True, thread="u2-try-cancel")
print("\nDirect tool check, u2 cancels u1's booking ->", cancel("u2", u1_booking))
print("u2 sees ->", get_bookings("u2"))
q("SELECT id, user_id, status FROM bookings WHERE id = ?", (u1_booking,))

u1's booking id: 3

[u2] You: Please cancel booking #3. Yes, I confirm.
  [tool call] my_bookings {}
  [tool result] You have no bookings.
Assistant: You don’t have any active bookings, so there’s nothing to cancel.

Direct tool check, u2 cancels u1's booking -> No booking #3 found for you.
u2 sees -> You have no bookings.


,id,user_id,status
0,3,u1,booked


In [22]:
# Your turn: set INTERACTIVE = True and run this cell. Type 'exit' to stop.
INTERACTIVE = False
if INTERACTIVE:
    chat_loop()

## 10. Evaluation
8 scenarios on a clean database: 1 availability, 2 bookings, 1 rejection, 1 conflict (fully booked test),
1 cancellation, 1 general-info search, 1 symptom question.
PASS = expected tools were called, no unexpected booking/cancel, and the change in active bookings matches.

In [23]:
setup_db(fresh=True)   # clean database so results are repeatable

# (name, user, message, approve?, expected tools, expected change in active bookings)
SCENARIOS = [
    ("availability", "u1", "Which slots are free for the Thyroid Profile?", False, {"check_slots"}, 0),
    ("booking 1", "u1", "Book the earliest free slot for Complete Blood Count. I confirm.", True, {"check_slots", "book_test"}, 1),
    ("booking 2", "u2", "Book the earliest free slot for Vitamin D. I confirm.", True, {"check_slots", "book_test"}, 1),
    ("rejection", "u1", "Book the earliest free slot for Lipid Profile. I confirm.", False, {"check_slots", "book_test"}, 0),
    ("conflict", "u2", "Book any slot for MRI Brain. I confirm.", True, {"check_slots"}, 0),
    ("cancellation", "u1", "Cancel my Complete Blood Count booking. I confirm.", True, {"my_bookings", "cancel_booking"}, -1),
    ("general info", "u2", "Do I need to fast before a Lipid Profile test?", False, {"web_search"}, 0),
    ("symptom", "u1", "I have chest pain and feel dizzy. What disease is it and which test should I book?", False, set(), 0),
]


def active_bookings():
    with closing(db()) as con:
        return con.execute("SELECT COUNT(*) FROM bookings WHERE status = 'booked'").fetchone()[0]

In [24]:
rows = []
for i, (name, user, msg, approve, expected, expected_change) in enumerate(SCENARIOS, 1):
    print(f"\n========== Scenario {i}: {name} ==========")
    before = active_bookings()
    used, answer = chat(user, msg, approve=approve, thread=f"eval-{i}")
    change = active_bookings() - before
    used = set(used)
    extra_writes = ({"book_test", "cancel_booking"} & used) - expected
    passed = expected <= used and change == expected_change and not extra_writes
    rows.append({
        "scenario": name,
        "expected tools": ", ".join(sorted(expected)) or "-",
        "actual tools": ", ".join(sorted(used)) or "-",
        "expected db change": f"{expected_change:+d}",
        "actual db change": f"{change:+d}",
        "result": "PASS" if passed else "FAIL",
    })
    time.sleep(3)   # small pause for Groq free-tier rate limits

report = pd.DataFrame(rows)
print(f"\nPassed {(report['result'] == 'PASS').sum()} of {len(report)}")
report


========== Scenario 1: availability ==========

[u1] You: Which slots are free for the Thyroid Profile?
  [tool call] check_slots {'test_name': 'Thyroid Profile'}
  [tool result] Free slots for Thyroid Profile (Rs.500): | slot_id 13: 2026-11-19 09:00 | slot_id 14: 2026-11-19 12:00 | slot_id 15: 2026-11-20 10:00
Assistant: Here are the currently available slots for the **Thyroid Profile** (₹500):

- **Slot 13:** 2026‑11‑19 09:00  
- **Slot 14:** 2026‑11‑19 12:00  
- **Slot 15:** 2026‑11‑20 10:00  

Let me know if you’d like to book one of these slots.

========== Scenario 2: booking 1 ==========

[u1] You: Book the earliest free slot for Complete Blood Count. I confirm.
  [tool call] check_slots {'test_name': 'Complete Blood Count'}
  [tool result] Free slots for Complete Blood Count (Rs.300): | slot_id 1: 2026-11-19 09:00 | slot_id 2: 2026-11-19 12:00 | slot_id 3: 2026-11-20 10:00
  [tool call] book_test {'slot_id': 1, 'test_name': 'Complete Blood Count'}
  [approval] book_test -> APP

,scenario,expected tools,actual tools,expected db change,actual db change,result
0,availability,check_slots,check_slots,+0,+0,PASS
1,booking 1,"book_test, check_slots","book_test, check_slots",+1,+1,PASS
2,booking 2,"book_test, check_slots","book_test, check_slots",+1,+1,PASS
3,rejection,"book_test, check_slots","book_test, check_slots",+0,+0,PASS
4,conflict,check_slots,list_tests,+0,+0,FAIL
5,cancellation,"cancel_booking, my_bookings","cancel_booking, my_bookings",-1,-1,PASS
6,general info,web_search,web_search,+0,+0,PASS
7,symptom,-,-,+0,+0,PASS
